This tutorial demonstrates a-to-z how to optimise Lennard Jones parameters for liquid argon, and without going into details. For details see other tuturials and wider MDMC documentation.

In [ ]:
# Imports used for this tutorial
import numpy as np
import os
from scipy.interpolate import interp2d
from MDMC.control import Control
from MDMC.MD import Atom, Dispersion, LennardJones, Simulation, Universe

In [ ]:
# Change the number of threads depending on the number of physical cores on your computer
# as it was tested for LAMMPS
os.environ["OMP_NUM_THREADS"] = "4"

In [ ]:
# Build universe with density 0.0176 atoms per AA^-3
density = 0.0176
# This means cubic universe of side:
# 23.0668 A will contain 216 Ar atoms
# 26.911 A will contain 343 Ar atoms
# 30.7553 A will contain 512 Ar atoms
# 38.4441 A will contain 1000 Ar atoms
universe = Universe(dimensions=23.0668)
Ar = Atom('Ar', charge=0.)
# Calculating number of Ar atoms needed to obtain density
n_ar_atoms = int(density * np.product(universe.dimensions))
print(f'Number of argon atoms = {n_ar_atoms}')
universe.fill(Ar, num_struc_units=(n_ar_atoms))

In the Jupyter cell above a box of Argon atoms are setup. However, at this point there is no interaction forces between the argon atoms! In the cell below an appropriate (for argon) force-field interaction potential is defined

In [ ]:
Ar_dispersion = Dispersion(universe,
                           (Ar.atom_type, Ar.atom_type),
                           cutoff=8.,
                           function=LennardJones(epsilon=1.0243, sigma=3.36))

In this case the interaction potential chosen to be the humble Lennard Jones (to get info see doc or type `help(LennardJones)`).

Also, a `cutoff` value is chosen (see `help(Dispersion)` for more info). A [role of thumb](https://en.wikipedia.org/wiki/Lennard-Jones_potential) is to pick `cutoff=2.5*sigma`. The value for argo n is recommended to be between 8 and 12 ang. `cutoff` is not a force-field parameter and therefore will not be refined. Ideally, and for any system you want to pick at value of the `cutoff` which is small while not compromising accuracy. For this system picking a value between 8 and 12 ang is found to give near identifical results.

Next and before starting the refinement set up the MD engine and equilibrate the system. Note with MDMC the equilibration is just needed to be done once. 

In [ ]:
# MD Engine setup
simulation = Simulation(universe,
                        engine="lammps",
                        time_step=10.0,#10.340088,#9.40008,
                        temperature=120.,
                        traj_step=25)

In [ ]:
# Energy Minimization and equilibration
simulation.minimize(n_steps=5000)
simulation.run(n_steps=10000, equilibration=True)

OK time to set up the actually refinement of the force-field parameters. 

First we need some data to refine against

In [ ]:
# exp_datasets is a list of dictionaries with one dictionary per experimental
# dataset
exp_datasets = [{'file_name':'data/Well_s_q_omega_Ar_data.xml',
                 'type':'SQw',
                 'reader':'xml_SQw',
                 'weight':1.,
                 'auto_scale':True}]

In [ ]:
# Fit parameters is the set of all unique fit parameters in the universe
# which are not fixed.
fit_parameters = set([p for p in universe.parameters if p.fixed is False])

control = Control(simulation=simulation,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  MC_norm=1,
                  minimizer_type="MMC",
                  reset_config=False,
                  MD_steps=2000,
                  energy_resolution=8.)

In [ ]:
# Well's Argon file seem to have incorrectly labelled f as w, so correct for
# this by multiplying E by 2pi
exp_obs = control.observable_pairs[0].exp_obs
Q = exp_obs.Q
E = exp_obs.E * 2 * np.pi
# copy the updated E values, and Q values back to the control.observable
control.observable_pairs[0].exp_obs.independent_variables = {'E':E_uniform,
                                                             'Q':Q_uniform}
control.observable_pairs[0].MD_obs.independent_variables = {'E':E_uniform,
                                                            'Q':Q_uniform}

And finally start the refinement! Bump up `n_steps` from 3 when you are ready.

In [ ]:
# Run the refinement, i.e. refine the FF parameters against the data
control.refine(n_steps=3)

So where do you from here. For steps to investigate the results from the refinement in more detail see the tutorial mdmc_tutorials/creating-an-observable.ipynb